# Assignment 2: Story & Insights
### Jordan Cars Market 2026 – A Data Story

**Author:** Luiza Stepanyan

---

## The Story

> **What does the Jordanian second-hand car market reveal about affordability, consumer preferences, and how it compares to the American market?**

In this notebook, I will move from raw exploration to a focused narrative. After completing our EDA, three key themes emerged:

1. **Affordability & Pricing** — Most cars are concentrated in the low-price range, driven by high mileage and older models.
2. **Fuel Type Trends** — Hybrid and electric cars command premium prices, signaling a market in transition.
3. **Geography** — Amman dominates the market; regional disparities tell a deeper economic story.

As a bonus, I will enrich the story by **comparing Jordan's market with the USA**, using a comparable used car dataset.

In [203]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# I will set up a color palette to consistently use throughout the project
JORDAN_COLOR = '#2196F3'   # blue - Jordan data
USA_COLOR    = '#FF5722'   # red - USA data
ACCENT       = '#4CAF50'   # green - highlights

_I will reproduce all cleaning steps from HW1 in one place so that the flow of the logic is clear._

In [204]:
import kagglehub, os

path = kagglehub.dataset_download('alzyood95/jordan-cars-market-2026')
df_jordan = pd.read_csv(os.path.join(path, 'cars.csv'))

df_jordan['Price_JOD'] = (
    df_jordan['Price']
    .str.replace(' JOD', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

df_jordan['Mileage_clean'] = df_jordan['Mileage'].str.replace(r'[^\d\-]', '', regex=True)
df_jordan.loc[df_jordan['Mileage_clean'] == '', 'Mileage_clean'] = '0'
split_vals = df_jordan['Mileage_clean'].str.split('-', expand=True)
df_jordan['Mileage_From'] = pd.to_numeric(split_vals[0], errors='coerce')
df_jordan['Mileage_To']   = pd.to_numeric(split_vals[1], errors='coerce')
df_jordan['Mileage_To']   = df_jordan['Mileage_To'].fillna(df_jordan['Mileage_From'])
df_jordan['Mileage_Avg']  = (df_jordan['Mileage_From'] + df_jordan['Mileage_To']) / 2
df_jordan.drop(columns=['Price', 'Mileage', 'Mileage_clean'], inplace=True)
df_jordan = df_jordan[df_jordan['Year'] != df_jordan['Year'].min()]
df_jordan['Brand'] = df_jordan['Model'].str.split().str[0]
df_jordan['City']  = df_jordan['Location'].str.split(',').str[0]
for idx in df_jordan[df_jordan['Price_JOD'].isna()].index:
    row = df_jordan.loc[idx]
    similar = df_jordan[
        (df_jordan['Model'] == row['Model']) &
        (df_jordan['Year']  == row['Year'])  &
        (df_jordan['Condition'] == row['Condition']) &
        (df_jordan['Price_JOD'].notna())
    ]
    df_jordan.loc[idx, 'Price_JOD'] = similar['Price_JOD'].median() if not similar.empty else df_jordan['Price_JOD'].median()

# I will also add a  USD conversion column for further analysis, I looked up, 1 JOD is almost 1.41 USD as of 2026 April
JOD_TO_USD = 1.41
df_jordan['Price_USD'] = df_jordan['Price_JOD'] * JOD_TO_USD

print(f'Jordan dataset: {df_jordan.shape[0]:,} rows × {df_jordan.shape[1]} columns')
df_jordan.head(3)

Jordan dataset: 2,093 rows × 17 columns


,ID,Model,Year,Condition,Fuel Type,Seller Type,Location,Insurance,Transmission,Color,Price_JOD,Mileage_From,Mileage_To,Mileage_Avg,Brand,City,Price_USD
0,1,Toyota Camry GLE,2023,used,Hybrid,subjective,"Ramtha, Ramtha",No insurance,Automatic,white,22500.0,70000,79999.0,74999.5,Toyota,Ramtha,31725.00
1,2,BMW X5 Series xDrive40i,2020,used,gasoline,undefined,"Amman, the chair",No insurance,Automatic,beige,64399.0,100000,109999.0,104999.5,BMW,Amman,90802.59
2,3,Mercedes-Benz SS-Class 320,2020,New (Zero),gasoline,an agency,"Amman, Abdoun",No insurance,Automatic,brown,9500.0,40000,49999.0,44999.5,Mercedes-Benz,Amman,13395.00


I use the **US Used Cars Dataset** from Kaggle (`austinreese/craigslist-carstrucks-data`), which contains used car listings scraped from Craigslist - the closest equivalent to Jordan's second-hand market.

In [205]:
# USA dataset
path_usa = kagglehub.dataset_download('austinreese/craigslist-carstrucks-data')
print('Files:', os.listdir(path_usa))

Files: ['vehicles.csv']


In [206]:
df_usa_raw = pd.read_csv(os.path.join(path_usa, 'vehicles.csv'),
                         usecols=['price', 'year', 'manufacturer', 'model',
                                  'condition', 'fuel', 'odometer', 'state', 'lat', 'long'],
                         low_memory=False)

# Basic cleaning
df_usa = df_usa_raw.copy()
df_usa = df_usa.rename(columns={
    'price': 'Price_USD', 'year': 'Year', 'manufacturer': 'Brand',
    'fuel': 'Fuel_Type', 'odometer': 'Mileage_miles', 'condition': 'Condition',
    'state': 'State'
})

# Removing nonsensical prices to keep $500 – $150,000
df_usa = df_usa[(df_usa['Price_USD'] >= 500) & (df_usa['Price_USD'] <= 150_000)]

# Removing null years / mileage
df_usa = df_usa.dropna(subset=['Year', 'Mileage_miles', 'Price_USD', 'Fuel_Type'])
df_usa['Year'] = df_usa['Year'].astype(int)
df_usa = df_usa[df_usa['Year'] >= 1990]

# Converting miles to km for comparison
df_usa['Mileage_km'] = df_usa['Mileage_miles'] * 1.60934

# also standardizing fuel names to match Jordan
fuel_map = {'gas': 'Gasoline', 'diesel': 'Diesel', 'hybrid': 'Hybrid',
            'electric': 'Electric', 'other': 'Other'}
df_usa['Fuel_Type'] = df_usa['Fuel_Type'].map(fuel_map).fillna('Other')

# sample for performance (keeping 20k rows)
df_usa_sample = df_usa.sample(20_000, random_state=42)

print(f'USA dataset (sampled): {df_usa_sample.shape[0]:,} rows × {df_usa_sample.shape[1]} columns')
df_usa_sample.head(3)

USA dataset (sampled): 20,000 rows × 11 columns


,Price_USD,Year,Brand,model,Condition,Fuel_Type,Mileage_miles,State,lat,long,Mileage_km
213046,29990,2016,ford,f-450 super duty,excellent,Gasoline,173470.0,mn,47.920244,-97.044974,279172.2098
303755,3950,2007,hyundai,sonata gls,good,Gasoline,127000.0,oh,39.927400,-82.004100,204386.1800
158669,20000,2014,chevrolet,silverado 1500,NaN,Gasoline,120000.0,ia,41.379465,-93.045959,193120.8000


In [207]:
df_usa["Fuel_Type"].unique()

array(['Gasoline', 'Other', 'Diesel', 'Hybrid', 'Electric'], dtype=object)

In [208]:
df_jordan["Fuel Type"].unique()

array(['Hybrid', 'gasoline', 'electricity', 'undefined', 'diesel'],
      dtype=object)

In [209]:
# I will also use standardizing 

fuel_mapping = {
    'Gasoline': 'gasoline',
    'Electric': 'electricity',
    'Diesel': 'diesel',
    'Hybrid': 'hybrid',
    'Other': 'undefined', 
    
}

df_usa['Fuel_Type'] = df_usa['Fuel_Type'].replace(fuel_mapping)
df_jordan['Fuel Type'] = df_jordan['Fuel Type'].replace(fuel_mapping)
# Renaming column in Jordan dataset
df_jordan = df_jordan.rename(columns={'Fuel Type': 'Fuel_Type'})

In [210]:
# further standartization based on their features
fuel_rename_lower = {
    'gasoline': 'gasoline',
    'diesel': 'diesel',
    'hybrid': 'hybrid',
    'electricity': 'electricity',
    'undefined': 'undefined',
    'gas': 'gasoline',      
    'electric': 'electricity',
    'other': 'undefined'     
}

# applying lowercase also  mapping
df_jordan['Fuel_Type'] = df_jordan['Fuel_Type'].str.lower().map(fuel_rename_lower)
df_usa_sample['Fuel_Type'] = df_usa_sample['Fuel_Type'].str.lower().map(fuel_rename_lower)

#defining an order for further correspondance
fuel_order = ['gasoline', 'diesel', 'hybrid', 'electricity', 'undefined']

---
## Part 1 — Affordability & Pricing

### The EDA revealed a sharp right-skewed price distribution. Here we make this insight interactive and compare it to the USA.

In [211]:
# Figure 1: Price Distribution – Jordan vs USA for which I will use overlapping histograms

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df_jordan['Price_USD'],
    nbinsx=60,
    name='Jordan',
    opacity=0.7,
    marker_color=JORDAN_COLOR,
    histnorm='percent'
))

fig.add_trace(go.Histogram(
    x=df_usa_sample['Price_USD'],
    nbinsx=60,
    name='USA',
    opacity=0.7,
    marker_color=USA_COLOR,
    histnorm='percent'
))

fig.update_layout(
    barmode='overlay',
    title='<b>Price Distribution: Jordan vs USA</b><br><sub>Both markets normalised to % of listings — USD equivalent</sub>',
    xaxis_title='Price (USD)',
    yaxis_title='% of Listings',
    xaxis_range=[0, 80_000],
    legend=dict(x=0.75, y=0.95),
    template='plotly_white',
    height=450
)

# adding annotation for Jordan median
jordan_median_usd = df_jordan['Price_USD'].median()
fig.add_vline(x=jordan_median_usd, line_dash='dash', line_color=JORDAN_COLOR,
              annotation_text=f'JO median: ${jordan_median_usd:,.0f}',
              annotation_position='top right')

usa_median = df_usa_sample['Price_USD'].median()
fig.add_vline(x=usa_median, line_dash='dash', line_color=USA_COLOR,
              annotation_text=f'US median: ${usa_median:,.0f}',
              annotation_position='top left')

fig.show()

From the plot, we can see that both distributions are right-skewed, meaning that most cars are concentrated in the lower price ranges, while fewer listings exist at higher prices. However, there are some noticeable differences between the two markets.
In Jordan, the distribution appears slightly more spread out, with a visible presence of listings across a wider price range, including higher-end vehicles. In contrast, the USA distribution is more concentrated in the lower to mid-price ranges, with a sharper peak at lower prices and fewer high-priced outliers.
The median price lines help clarify this difference. The median price in Jordan (around 17,000 US dollars) is slightly higher than in the USA (around 16,000 US dollars). This suggests that, on average, cars listed in Jordan tend to be a bit more expensive than those in the sampled US dataset.
Another important observation is the long tail extending toward higher prices (up to around 80,000 US dollars), especially for Jordan. This indicates the presence of luxury or premium vehicles, though they make up a small percentage of total listings.
Overall, the chart shows that while both markets share a similar general shape, Jordan’s market leans slightly toward higher prices and greater variability, whereas the US market is more concentrated in the lower price segments.

In [213]:
# Figure 2: Median Price by Year Jordan vs USA

jordan_by_year = df_jordan.groupby('Year')['Price_USD'].median().reset_index()
usa_by_year    = df_usa_sample.groupby('Year')['Price_USD'].median().reset_index()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=jordan_by_year['Year'], y=jordan_by_year['Price_USD'],
    mode='lines+markers', name='Jordan',
    line=dict(color=JORDAN_COLOR, width=2),
    marker=dict(size=6)
))

fig2.add_trace(go.Scatter(
    x=usa_by_year['Year'], y=usa_by_year['Price_USD'],
    mode='lines+markers', name='USA',
    line=dict(color=USA_COLOR, width=2),
    marker=dict(size=6)
))

fig2.update_layout(
    title='<b>Median Car Price by Model Year</b><br><sub>How does car age affect price in Jordan vs the USA?</sub>',
    xaxis_title='Model Year',
    yaxis_title='Median Price (USD)',
    template='plotly_white',
    legend=dict(x=0.05, y=0.95),
    height=450
)

fig2.show()

From the visualization above, we can observe that prices declined after the year 2020 for both countries, which is an intuitive trend considering that the COVID-19 pandemic occurred around this time, meaning that car production was likely disrupted, resulting in fewer vehicles being manufactured during this period. Additionally, we can see that cars manufactured before the year 2000 had lower median prices in Jordan compared to the USA. Based on my research, several factors may explain this discrepancy. The primary reasons include lower purchasing power in Jordan, higher maintenance costs for aging vehicles, limited demand for older automotive technology, and harsh environmental conditions that accelerate wear and tear, all of which reduce the value of older cars and make them less desirable and cheaper in the Jordanian market. Another possible reason is that Jordan has very high customs and taxes on newer or larger-engine vehicles, which distorts the market and makes older, cheaper cars more common in the used market.

Looking at cars produced between 2010 and 2020, we see they are significantly cheaper in Jordan than in the USA. This can be explained by the fact that Jordan relies heavily on importing used cars from markets like the USA, Japan, and Europe, meaning that the supply of these vehicles is abundant, which drives prices down. At the same time, Jordanian buyers have lower average incomes compared to American buyers, so the demand for relatively newer but still affordable cars is price-sensitive, keeping median prices low. In contrast, in the USA, cars from the same 2010-2020 period are still considered relatively modern and reliable, and there is strong domestic demand for them, which supports higher prices.

Before this period, prices were relatively close between the two markets, likely because very old cars (pre-2010) have limited value in both countries, so their prices converge at a low baseline. After approximately 2023, prices in the USA dropped noticeably, and we can also note that the US dataset contains fewer modern vehicle listings compared to the Jordanian dataset.

In [214]:
The above visualization is actually very intuitive. We can clearly observe that the mileage for electric vehicles is significantly lower compared to cars powered by other fuel types. The main reason for this is that electric cars entered mass production much later than gasoline, diesel, and hybrid vehicles, meaning they simply have not been on the road long enough to accumulate high mileage. This is especially evident when looking at the electric vehicle points on the scatter plot, which are clustered in the low-mileage range.

However, the key insight from this visualization is that hybrid cars appear to be among the most valuable vehicles in the Jordanian used car market. They consistently command higher prices across almost all mileage ranges compared to gasoline and diesel cars. This suggests that Jordanian buyers recognize and are willing to pay a premium for the fuel efficiency and lower running costs that hybrid technology offers, even on the second-hand market.

As expected, we can also see a very clear and strong negative relationship between mileage and price across all fuel types. As mileage increases, prices decrease drastically, which is again very intuitive. This inverse relationship reflects the fundamental principle that higher usage leads to more wear and tear, bringing the car closer to the end of its reliable lifespan. Interestingly, we notice that diesel cars, while fewer in number compared to gasoline vehicles, tend to be scattered across both low and high price points, possibly reflecting a mix of older commercial vehicles and newer passenger diesels.

The above visualization is actually very intuitive. We can clearly observe that the mileage for electric vehicles is significantly lower compared to cars powered by other fuel types. The main reason for this is that electric cars entered mass production much later than gasoline, diesel, and hybrid vehicles, meaning they simply have not been on the road long enough to accumulate high mileage. This is especially evident when looking at the electric vehicle points on the scatter plot, which are clustered in the low-mileage range.

However, the key insight from this visualization is that hybrid cars appear to be among the most valuable vehicles in the Jordanian used car market. They consistently command higher prices across almost all mileage ranges compared to gasoline and diesel cars. This suggests that Jordanian buyers recognize and are willing to pay a premium for the fuel efficiency and lower running costs that hybrid technology offers, even on the second-hand market.

As expected, we can also see a very clear and strong negative relationship between mileage and price across all fuel types. As mileage increases, prices decrease drastically, which is again very intuitive. This inverse relationship reflects the fundamental principle that higher usage leads to more wear and tear, bringing the car closer to the end of its reliable lifespan. Interestingly, we notice that diesel cars, while fewer in number compared to gasoline vehicles, tend to be scattered across both low and high price points, possibly reflecting a mix of older commercial vehicles and newer passenger diesels.

---
## Part 2 — Fuel Type Transition
### Story: Jordan's market is slowly shifting toward greener fuels, but gasoline still dominates


In [215]:

# Computing fuel percentages
jordan_fuel = df_jordan['Fuel_Type'].value_counts(normalize=True).mul(100).round(1).reset_index()
jordan_fuel.columns = ['Fuel_Type', 'Pct']

usa_fuel = df_usa_sample['Fuel_Type'].value_counts(normalize=True).mul(100).round(1).reset_index()
usa_fuel.columns = ['Fuel_Type', 'Pct']

# Standardizing order and including all categories for safety
fuel_order = ['gasoline', 'diesel', 'hybrid', 'electricity', 'undefined']

# Reindexing to ensure all categories exist
jordan_fuel = jordan_fuel.set_index('Fuel_Type').reindex(fuel_order, fill_value=0).reset_index()
usa_fuel    = usa_fuel.set_index('Fuel_Type').reindex(fuel_order, fill_value=0).reset_index()

# Replacing 0 with tiny positive to ensure slice renders
jordan_fuel['Pct_plot'] = jordan_fuel['Pct'].replace(0, 0.0001)
usa_fuel['Pct_plot']    = usa_fuel['Pct'].replace(0, 0.0001)

# Assigning consistent colors
fuel_colors = {
    'gasoline': '#636EFA',  # blue
    'diesel': '#EF553B',    # red
    'hybrid': '#00CC96',    # green
    'electricity': '#AB63FA',  # purple
    'undefined': '#FFA15A'      # orange
}


fig4 = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'domain'}, {'type': 'domain'}]],
    subplot_titles=['Jordan', 'USA']
)

# Jordan
fig4.add_trace(go.Pie(
    labels=jordan_fuel['Fuel_Type'],
    values=jordan_fuel['Pct_plot'],
    hole=0.4,
    name='Jordan',
    marker_colors=[fuel_colors[f] for f in jordan_fuel['Fuel_Type']],
    hovertemplate='%{label}: %{value:.1f}%<extra></extra>'
), row=1, col=1)

# USA
fig4.add_trace(go.Pie(
    labels=usa_fuel['Fuel_Type'],
    values=usa_fuel['Pct_plot'],
    hole=0.4,
    name='USA',
    marker_colors=[fuel_colors[f] for f in usa_fuel['Fuel_Type']],
    hovertemplate='%{label}: %{value:.1f}%<extra></extra>'
), row=1, col=2)

fig4.update_layout(
    title='<b>Fuel Type Market Share: Jordan vs USA</b>',
    template='plotly_white',
    height=420
)

fig4.show()

From this visualization , I actually observed something quite surprising to me. I initially assumed that the United States would have a significantly higher proportion of electric vehicles, but the data shows that only 0.4% of cars in the US sample are electric. However, taking into consideration that the latest model year in the US dataset is 2022, this finding becomes more understandable. The major electric vehicle boom, driven by increased production from Tesla, the launch of the Ford Mustang Mach-E, the Chevrolet Bolt, and growing adoption of models from Hyundai, Kia, and Volkswagen, largely accelerated in 2022 and especially after 2023. Since our US data does not extend beyond 2022, we are essentially capturing the market right before the EV explosion, which explains the low percentage.

On the other hand, the Jordanian market reveals a much more interesting story. Hybrid cars in Jordan account for nearly 34% of the market, which is remarkably high and almost comparable to gasoline cars at 44%. This is fascinating for several reasons. First, it suggests that Jordanian consumers have actively embraced hybrid technology as a practical middle-ground solution, valuing the fuel savings and lower emissions without the charging infrastructure concerns that still challenge full electric vehicles. Second, Jordan's fuel prices are significantly higher than in the US, which creates a strong economic incentive for buyers to prioritize fuel-efficient vehicles like hybrids. Third, the Jordanian government has historically offered customs and tax incentives for hybrid vehicles, making them more affordable and attractive compared to traditional gasoline cars.

When we compare the two markets directly, a clear contrast emerges. The United States remains overwhelmingly dependent on gasoline, which constitutes over 84% of the market, reflecting a culture of large engines, lower fuel prices, and slower adoption of alternative fuel technologies until very recently. Diesel also maintains a small but notable presence in the US at nearly 7%, primarily in pickup trucks and commercial vehicles. In Jordan, by contrast, the fuel mix is far more diversified. Beyond the strong hybrid presence, electric vehicles have already captured nearly 15% of the market, despite the US dataset showing only 0.4% for a similar time period. This suggests that Jordan may actually be adopting newer automotive technologies faster than the US, possibly due to government policies, higher fuel costs, and a more pragmatic consumer approach to total cost of ownership.

What connects these two markets is that both are ultimately transitioning away from pure gasoline dependence, but they are doing so at different speeds and through different pathways. The United States is skipping hybrids to some extent and moving directly toward electric vehicles, though this shift is only visible in data from 2023 onward. Jordan, constrained by less developed charging infrastructure and higher electricity costs in some areas, has embraced hybrids as an immediate and practical solution while still building a foundation for electric vehicle adoption. Interestingly, the "undefined" category in both markets, which likely represents missing or unspecified fuel type data, is much higher in the US at over 7% compared to only 3% in Jordan, suggesting differences in data completeness between the two datasets. Overall, the Jordanian market appears more diversified and forward-looking in its fuel mix than I initially expected, challenging the assumption that developing markets lag behind the US in adopting greener automotive technologies.

In [220]:
df_usa["Year"].max()

2022

In [216]:
# Figure 5: Fuel type adoption over years in Jordan

fuel_year = (
    df_jordan[df_jordan['Year'] >= 2000]
    .groupby(['Year', 'Fuel_Type'])
    .size()
    .reset_index(name='Count')
)

# Normalizing to % within each year
fuel_year['Pct'] = fuel_year.groupby('Year')['Count'].transform(lambda x: x / x.sum() * 100)

fig5 = px.area(
    fuel_year,
    x='Year', y='Pct', color='Fuel_Type',
    title='<b>Fuel Type Share by Year (Jordan, 2000–present)</b><br><sub>How the composition of new listings changed over time</sub>',
    labels={'Pct': '% of Listings', 'Year': 'Model Year', 'Fuel_Type': 'Fuel'},
    template='plotly_white',
    color_discrete_map=fuel_colors,  # consistent colors
    height=450
)

fig5.update_layout(legend_title_text='Fuel Type')
fig5.show()

From the visualization above, we can see how Jordan's car market has transformed dramatically since 2000. In the early 2000s, gasoline completely dominated the market, with only a small presence of diesel. The first major shift began around 2010 when hybrids started appearing, and their share grew steadily over the next decade. However, the most remarkable change happened after 2020. Between 2020 and 2025, gasoline dropped from being the majority to falling below 20%, while hybrids surged to become the market leader, peaking at over 60% of listings. Electric vehicles also emerged during this period, reaching around 25-30% of the market by 2024. This rapid transition away from gasoline in just a few years is striking and suggests that factors such as government incentives, rising fuel prices, or increased availability of hybrid and electric models have strongly influenced Jordanian consumers. Diesel has remained a small and stable niche throughout, never exceeding 10% of listings. Overall, the data shows that Jordan has undergone an extraordinary shift from a gasoline-dominated market to one where hybrids and electric vehicles now represent the vast majority of car listings.

In [217]:

#Figure 6: Median Price by Fuel Type: Jordan vs USA

# Jordan median prices
jordan_fuel_price = df_jordan.groupby('Fuel_Type')['Price_USD'].median().reindex(fuel_order).reset_index()
jordan_fuel_price['Market'] = 'Jordan'

# USA median prices
usa_fuel_price = df_usa_sample.groupby('Fuel_Type')['Price_USD'].median().reindex(fuel_order).reset_index()
usa_fuel_price['Market'] = 'USA'

# combining datasets
fuel_price_combined = pd.concat([jordan_fuel_price, usa_fuel_price], ignore_index=True)

# converting Price_USD to numeric for safety reasons
fuel_price_combined['Price_USD'] = pd.to_numeric(fuel_price_combined['Price_USD'], errors='coerce')

fig6 = px.bar(
    fuel_price_combined,
    x='Fuel_Type', y='Price_USD', color='Market',
    barmode='group',
    color_discrete_map={'Jordan': JORDAN_COLOR, 'USA': USA_COLOR},
    title='<b>Median Price by Fuel Type: Jordan vs USA</b>',
    labels={'Price_USD': 'Median Price (USD)', 'Fuel_Type': 'Fuel Type'},
    template='plotly_white',
    height=420
)
fig6.show()

From the visualization above, we can see some interesting differences between the two markets. In Jordan, hybrid cars have the highest median price at around 21,000 dollars, followed closely by electric vehicles at approximately 26,000 dollars, though the electric sample size is smaller. Gasoline and diesel cars are significantly cheaper in Jordan, with median prices around 10,000  and 11,000 dollars respectively. In the USA, the pattern is quite different. Diesel cars command the highest median price at nearly  32,000 USD, followed by electric vehicles at around 29,000 dollars and hybrids at approximately 13,000 dollars. Gasoline cars in the US have a median price of about  14,000 USD, which is actually higher than in Jordan. This contrast suggests that in Jordan, hybrids are perceived as premium vehicles worth paying extra for, possibly due to fuel savings and tax incentives. In the US, diesel commands a premium likely because it is associated with heavy-duty trucks and larger vehicles. Another notable observation is that electric vehicles are priced similarly in both markets around 26,000-29,000 USD, despite the much lower adoption rate in the US at the time of this data. The undefined category, representing missing fuel type data, shows higher median prices in both markets, which may indicate that these unspecified vehicles tend to be higher-end models where fuel type was less relevant to the listing. Overall, the key insight is that hybrid vehicles hold significantly more value in Jordan than in the US, reflecting different consumer preferences and market conditions.

---
## Part 3 — Brand Landscape
###  Story: Japanese brands dominate affordability; European brands signal status

In [221]:
# Figure 7: Top brands by listing count — Jordan

top_brands = df_jordan['Brand'].value_counts().head(12).reset_index()
top_brands.columns = ['Brand', 'Count']
top_brands['Brand'] = top_brands['Brand'].str.title()

fig7 = px.bar(
    top_brands.sort_values('Count'),
    x='Count', y='Brand',
    orientation='h',
    title='<b>Most Listed Car Brands in Jordan</b>',
    labels={'Count': 'Number of Listings', 'Brand': ''},
    color='Count',
    color_continuous_scale='Blues',
    template='plotly_white',
    height=460
)
fig7.update_coloraxes(showscale=False)
fig7.show()

we can see that Hyundai is the most listed car brand in Jordan with 242 listings, followed very closely by Kia with 225 listings and Toyota with 224 listings. This near three-way tie at the top is interesting because it suggests that Korean brands (Hyundai and Kia) are just as popular as the globally dominant Toyota in the Jordanian market. Mercedes-Benz comes in fourth with 181 listings, which is notably high for a luxury brand and indicates that German cars have a strong presence in Jordan. BYD, a Chinese electric vehicle manufacturer, appears in fifth place with 148 listings, which is remarkable considering that electric vehicles are still a relatively new technology and BYD has only recently expanded globally. This suggests that Jordanian buyers are open to Chinese EV brands. Ford, Volkswagen, BMW, Mitsubishi, Nissan, Land, and Honda round out the top 12 with between 64 and 101 listings each. Overall, the market shows a healthy mix of Asian, European, and American brands, with Korean and Japanese brands dominating the top spots. The presence of BYD so high on the list also reflects the rapid shift toward electric vehicles that we observed in earlier visualizations.

In [222]:
# Figure 8: Price vs Year interactive bubble by brand (Jordan)

top_brand_names = df_jordan['Brand'].value_counts().head(10).index.tolist()
df_brands = df_jordan[df_jordan['Brand'].isin(top_brand_names)].copy()
df_brands['Brand'] = df_brands['Brand'].str.title()

brand_summary = (
    df_brands.groupby(['Brand', 'Year'])
    .agg(Median_Price=('Price_USD', 'median'), Count=('Price_USD', 'count'))
    .reset_index()
)

fig8 = px.scatter(
    brand_summary[brand_summary['Year'] >= 2005],
    x='Year', y='Median_Price',
    size='Count', color='Brand',
    hover_name='Brand',
    hover_data={'Count': True, 'Median_Price': ':,.0f'},
    title='<b>Brand × Year: Median Price & Listing Volume</b><br><sub>Bubble size = number of listings</sub>',
    labels={'Median_Price': 'Median Price (USD)', 'Year': 'Model Year'},
    template='plotly_white',
    height=520,
    color_discrete_sequence=px.colors.qualitative.Vivid
)
fig8.show()

Here, Toyota and Hyundai show a relatively stable upward trend in median prices as model years become newer, with both brands having large bubbles in recent years, indicating high listing volumes for newer models. Kia follows a similar pattern but with slightly lower prices. Mercedes-Benz and BMW stand out as the premium brands, consistently commanding higher median prices across all years compared to Japanese and Korean brands. Their bubbles are generally smaller, reflecting lower listing volumes, which makes sense given that luxury cars are less common in the used market. BYD is a fascinating case, appearing only in very recent years (2022-2025) with rapidly growing bubble sizes and competitive pricing, which aligns with the brand's recent entry into the Jordanian market and the surge in electric vehicle adoption we observed earlier. Ford and Nissan show more moderate pricing, sitting between the economy brands and the luxury European brands. Volkswagen maintains a consistent mid-range price position across most years. Overall, the chart clearly shows that newer cars command higher prices for every brand, which is expected, and that luxury European brands maintain their premium positioning even in the used market. The growing bubble sizes for BYD in the most recent years is particularly noteworthy as it signals increasing market penetration for this Chinese EV brand in Jordan.

In [223]:
# Figure 9: Jordan vs USA — Top brands median price comparison

SHARED_BRANDS = ['Toyota', 'Kia', 'Hyundai', 'Honda', 'Bmw', 'Nissan', 'Mercedes-Benz']

df_jordan['Brand_title'] = df_jordan['Brand'].str.title()
df_usa_sample['Brand_title'] = df_usa_sample['Brand'].str.title()

jordan_bp = (df_jordan[df_jordan['Brand_title'].isin(SHARED_BRANDS)]
             .groupby('Brand_title')['Price_USD'].median().reset_index())
jordan_bp['Market'] = 'Jordan'

usa_bp = (df_usa_sample[df_usa_sample['Brand_title'].isin(SHARED_BRANDS)]
          .groupby('Brand_title')['Price_USD'].median().reset_index())
usa_bp['Market'] = 'USA'

brand_compare = pd.concat([jordan_bp, usa_bp])

fig9 = px.bar(
    brand_compare,
    x='Brand_title', y='Price_USD', color='Market',
    barmode='group',
    color_discrete_map={'Jordan': JORDAN_COLOR, 'USA': USA_COLOR},
    title='<b>Brand Median Prices: Jordan vs USA</b><br><sub>Same brands, different markets — how do prices compare?</sub>',
    labels={'Price_USD': 'Median Price (USD)', 'Brand_title': 'Brand'},
    template='plotly_white',
    height=440
)
fig9.show()

We can see that for every single brand in the comparison, median prices in Jordan are higher than in the USA. This is a striking finding because it contradicts the overall market trend we observed earlier, where the general median price in Jordan was lower than in the USA. BMW shows the largest gap, with Jordanian buyers paying nearly 34,000 dollars compared to around USD 19,000 in the US, which is almost double. Mercedes-Benz follows a similar pattern at around USD 29,000 in Jordan versus USD 20,000 in the US. Toyota, Kia, Hyundai, Honda, and Nissan all show consistent premiums in Jordan ranging from roughly USD 2,000 to 7,000 higher than their US counterparts.

This finding suggests that while the overall Jordanian market may have lower median prices due to a larger proportion of older, cheaper vehicles, the same specific brands actually cost more in Jordan than in the United States. Several factors could explain this. First, Jordan imposes high customs duties and taxes on imported cars, which drives up prices across the board. Second, the Jordanian market relies heavily on used car imports, and the most desirable brands like Toyota and Hyundai may command premium prices due to their reputation for reliability and lower maintenance costs. Third, the US market benefits from economies of scale, with massive local production and high competition keeping prices lower. Fourth, the specific mix of models within each brand may differ between the two countries, with Jordan potentially importing higher-trim or better-equipped versions of the same brands.

Overall, the key insight is that car buyers in Jordan pay significantly more for the same brands compared to American buyers, despite Jordan having a lower overall market median price. This highlights how market composition, import taxes, and consumer preferences can create counterintuitive pricing patterns across countries.

---
## Part 4 — Spatial Analysis (Jordan)
### Story: Where are cars listed in Jordan? How do prices vary by city? 

In [224]:
import folium
from folium.features import GeoJsonTooltip
from folium.plugins import HeatMap
import json
import requests
import geopandas as gpd
import matplotlib.pyplot as plt

# Downloading Jordan GeoJSON from reliable source
url = "https://raw.githubusercontent.com/apache/superset/master/superset-frontend/plugins/legacy-plugin-chart-country-map/src/countries/jordan.geojson"

response = requests.get(url)
with open("jordan.geojson", "wb") as f:
    f.write(response.content)

gdf = gpd.read_file("jordan.geojson")

with open("jordan.geojson", "r") as f:
    jordan_geojson = json.load(f)

print("GeoJSON loaded successfully!")
print(f"Columns: {gdf.columns.tolist()}")
print(f"Number of features: {len(gdf)}")
print(f"First feature properties: {gdf.iloc[0].to_dict()}")

location_field = None
for col in ['name', 'NAME_1', 'admin', 'governorate']:
    if col in gdf.columns:
        location_field = col
        break

if location_field is None:
    location_field = gdf.columns[0]  # Use first column as fallback

print(f"\nUsing '{location_field}' as location identifier")
print(f"Region names: {gdf[location_field].tolist()}")

GeoJSON loaded successfully!
Columns: ['ISO', 'NAME_1', 'geometry']
Number of features: 12
First feature properties: {'ISO': 'JO-IR', 'NAME_1': 'Irbid', 'geometry': <POLYGON ((35.574 32.554, 35.576 32.555, 35.58 32.56, 35.574 32.573, 35.571 ...>}

Using 'NAME_1' as location identifier
Region names: ['Irbid', 'Madaba', 'Karak', 'Tafilah', 'Aqaba', 'Balqa', 'Mafraq', 'Ma`an', 'Amman', 'Zarqa', 'Ajlun', 'Jarash']


In [226]:
# Defining coordinates for Jordanian cities (for marker placement)
city_coords = {
    'Amman': [31.945367, 35.928372],
    'Irbid': [32.556847, 35.847896],
    'Zarqa': [32.072915, 36.097947],
    'Ramtha': [32.558889, 36.006944],
    'Salt': [32.039167, 35.727222],
    'Madaba': [31.716667, 35.800000],
    'Aqaba': [29.531917, 35.005314],
    'Mafraq': [32.342222, 36.208333],
    'Jerash': [32.274167, 35.896389],
    'Karak': [31.185000, 35.704722],
    'Ajloun': [32.333333, 35.750000],
    'Tafilah': [30.833333, 35.600000],
    "Ma'an": [30.196667, 35.734167]
}

# Aggregating car listings by city
city_stats = df_jordan.groupby('City').agg({
    'Price_USD': ['count', 'median', 'mean'],
    'Mileage_Avg': 'median',
    'Year': 'median'
}).reset_index()

city_stats.columns = ['City', 'Listings', 'Median_Price', 'Mean_Price', 'Median_Mileage', 'Median_Year']
city_stats = city_stats.sort_values('Listings', ascending=False)

total_listings = city_stats['Listings'].sum()
city_stats['Pct_of_Listings'] = (city_stats['Listings'] / total_listings) * 100

print("\nTop 10 cities by number of listings:")
print(city_stats.head(10))

city_stats['lat'] = city_stats['City'].map(lambda x: city_coords.get(x, [31.2, 36.0])[0])
city_stats['lon'] = city_stats['City'].map(lambda x: city_coords.get(x, [31.2, 36.0])[1])

city_stats = city_stats.dropna(subset=['lat', 'lon'])

print(f"\nProcessed {len(city_stats)} cities with coordinates")
print(f"Coverage: {city_stats['Listings'].sum():,} / {total_listings:,} listings ({city_stats['Listings'].sum()/total_listings*100:.1f}%)")


Top 10 cities by number of listings:
         City  Listings  Median_Price    Mean_Price  Median_Mileage  \
3       Amman      1494      17625.00  24602.454839         94999.5   
17      Zarqa       254      21925.50  22911.789449         44999.5   
6       Irbid       199      14100.00  17025.700402        124999.5   
14       Salt        27       9165.00  11953.666667        200000.0   
10     Madaba        18       9975.75  10101.083333        179999.5   
7      Jerash        16       9870.00  12500.531250        179999.5   
0      Ajloun        13       5076.00   9425.307692        200000.0   
8       Karak        11      13959.00  11767.090909        164999.5   
1   Al Mafraq        10       4653.00   5781.000000        184999.5   
11     Mafraq        10       5992.50   6937.200000        200000.0   

    Median_Year  Pct_of_Listings  
3        2018.0        71.380793  
17       2022.0        12.135690  
6        2016.0         9.507883  
14       2007.0         1.290014  
10   

From the data above, we can see that Amman is overwhelmingly the dominant city in the Jordanian car market, accounting for 1,494 listings which represents 71.4% of all cars in the dataset. This is expected since Amman is the capital and largest city by far. Zarqa comes in second with 254 listings or 12.1% of the market, followed by Irbid with 199 listings or 9.5%. Together, these three cities alone account for over 93% of all car listings in Jordan, which highlights how centralized the country's used car market is around its major urban centers.

Looking at median prices, we see some interesting variation. Zarqa has the highest median price at approximately USD 21,925, which is actually higher than Amman's median of USD 17,625. This could indicate that Zarqa has a higher proportion of newer or more premium vehicles, or perhaps it serves as a hub for car dealerships and imports. Irbid has a lower median price of around USD 14,100, making it more affordable than the capital. Smaller cities like Salt, Madaba, Jerash, Ajloun, Karak, and Mafraq have much lower listing volumes, each contributing less than 1.3% of the total market. Their median prices also tend to be lower, ranging from approximately USD 4,650 to USD 14,000, which likely reflects older vehicle fleets or different economic conditions in these areas. For example, Ajloun has a median year of 2001 and a median price of only USD 5,076, while Zarqa has a median year of 2022 and a median price of USD 21,925, clearly showing that newer cars command higher prices and tend to concentrate in wealthier or more commercial cities. The median mileage across cities is quite high, with most cities showing figures between 95,000 and 200,000 kilometers, which is typical for used car markets.

In [227]:

import plotly.express as px

print("Checking city_stats columns:")
print(city_stats[['City', 'Listings', 'Median_Price', 'Mean_Price', 'Pct_of_Listings', 'lat', 'lon']].head())

fig_map1 = px.scatter_mapbox(
    city_stats,
    lat='lat',
    lon='lon',
    size='Listings',
    color='Median_Price',
    hover_name='City',
    hover_data={
        'Listings': ':,d',           # Formatting as integer with commas
        'Median_Price': '$.2f',      # Formatting as USD with 2 decimals
        'Mean_Price': '$.2f',        # Formatting as USD with 2 decimals
        'Median_Mileage': ':.0f km', # Formatting as kilometers
        'Pct_of_Listings': ':.1f%',  # Formatting as percentage
        'lat': False,                # Hiding latitude
        'lon': False                 # Hiding longitude
    },
    zoom=6,
    center={'lat': 31.2, 'lon': 36.0},
    title='<b>Jordan Car Market: Listings by City</b><br><sub>Bubble size = number of listings | Color = median price</sub>',
    color_continuous_scale='Viridis',
    size_max=50,
    height=600,
    mapbox_style='open-street-map'
)

# Customize hover template further for cleaner display
fig_map1.update_traces(
    hovertemplate='<b>%{hovertext}</b><br>' +
                  'Listings: %{customdata[0]:,d}<br>' +
                  'Median Price: $%{customdata[1]:,.0f}<br>' +
                  'Mean Price: $%{customdata[2]:,.0f}<br>' +
                  'Median Mileage: %{customdata[3]:,.0f} km<br>' +
                  'Market Share: %{customdata[4]:.1f}%<extra></extra>'
)

fig_map1.update_layout(
    margin={'r': 0, 't': 40, 'l': 0, 'b': 0}
)

fig_map1.show()

Checking city_stats columns:
      City  Listings  Median_Price    Mean_Price  Pct_of_Listings        lat  \
3    Amman      1494      17625.00  24602.454839        71.380793  31.945367   
17   Zarqa       254      21925.50  22911.789449        12.135690  32.072915   
6    Irbid       199      14100.00  17025.700402         9.507883  32.556847   
14    Salt        27       9165.00  11953.666667         1.290014  32.039167   
10  Madaba        18       9975.75  10101.083333         0.860010  31.716667   

          lon  
3   35.928372  
17  36.097947  
6   35.847896  
14  35.727222  
10  35.800000  


we can clearly see the geographic concentration of the Jordanian used car market. Amman appears as the largest bubble by a significant margin, confirming that the capital dominates with over 70% of all listings. The bubble is colored in the mid-range of the Viridis scale, reflecting Amman's median price of approximately USD 17,600. Zarqa and Irbid appear as the next largest bubbles, though they are substantially smaller than Amman, which aligns with the data showing they represent only 12 percent  and 9.5 percent of the market respectively.

Looking at the color coding, we can observe some interesting geographic price patterns. Zarqa shows a darker green color, indicating its higher median price of around USD 21,900, making it the most expensive city in the dataset. This is surprising because Zarqa is often considered an industrial city, but the data suggests it may have a higher concentration of newer or more premium vehicles. Amman shows a medium green color, while Irbid appears lighter, reflecting its lower median price of approximately USD 14,100. The smaller cities scattered across the map, such as Salt, Madaba, Jerash, Ajloun, Karak, and the various Mafraq entries, appear as tiny bubbles with lighter colors, indicating both low listing volumes and lower median prices, typically below USD 10,000.

The spatial distribution of the map shows that the vast majority of car listings are concentrated in the northwestern part of the country, centered around Amman, Zarqa, and Irbid. This region forms the economic heart of Jordan and contains the majority of the population. Cities further south like Karak, Tafilah, Ma'an, and Aqaba have very few listings, which makes sense given their smaller populations and more remote locations. Aqaba, despite being a major port city and tourist destination, shows relatively low listing volume, which could indicate that cars are imported through Aqaba but then sold in Amman and the northern cities where the demand is highest.

Overall, this map effectively communicates that the Jordanian used car market is highly centralized, both in terms of listing volume and pricing power, with Amman, Zarqa, and Irbid forming the core of the market while the rest of the country represents only a small fraction of activity.

In [198]:
import folium
from folium.features import GeoJsonTooltip

# Creating mapping from cities to governorates (using NAME_1 values from GeoJSON)
# Based on the GeoJSON output, governorates are: Irbid, Madaba, Karak, Tafilah, Aqaba, Balqa, Mafraq, Ma`an, Amman, Zarqa, Ajlun, Jarash

city_to_governorate = {
    'Amman': 'Amman',
    'Irbid': 'Irbid',
    'Zarqa': 'Zarqa',
    'Ramtha': 'Irbid',
    'Salt': 'Balqa',
    'Madaba': 'Madaba',
    'Aqaba': 'Aqaba',
    'Mafraq': 'Mafraq',
    'Jerash': 'Jarash',  # GeoJSON uses 'Jarash'
    'Karak': 'Karak',
    'Ajloun': 'Ajlun',    # GeoJSON uses 'Ajlun'
    'Tafilah': 'Tafilah',
    "Ma'an": "Ma`an",     # GeoJSON uses 'Ma`an'
    'Al Mafraq': 'Mafraq'
}

city_stats['Governorate'] = city_stats['City'].map(city_to_governorate).fillna('Other')

governorate_stats = city_stats.groupby('Governorate').agg({
    'Listings': 'sum',
    'Median_Price': 'median',
    'Mean_Price': 'mean'
}).reset_index()

print("\nStatistics by governorate:")
print(governorate_stats)

m_choropleth = folium.Map(location=[31.2, 36.0], zoom_start=7, tiles='CartoDB positron')

choropleth = folium.Choropleth(
    geo_data=jordan_geojson,
    data=governorate_stats,
    columns=['Governorate', 'Listings'],
    key_on='feature.properties.NAME_1',  # Using NAME_1 as the key
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name='Number of Car Listings',
    highlight=True,
    name='Car Listings by Governorate'
).add_to(m_choropleth)

folium.GeoJsonTooltip(
    fields=['NAME_1'],
    aliases=['Governorate:'],
    style='background-color: white; border: 1px solid black; border-radius: 5px; padding: 5px; font-family: Arial;'
).add_to(choropleth.geojson)

for _, row in city_stats.head(15).iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=min(row['Listings'] ** 0.35, 25),
        popup=f"""
            <b>{row['City']}</b><br>
            Listings: {row['Listings']:,}<br>
            Median Price: ${row['Median_Price']:,.0f}<br>
            Median Mileage: {row['Median_Mileage']:,.0f} km<br>
            Share: {row['Pct_of_Listings']:.1f}%
        """,
        color='darkblue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.5,
        weight=1.5
    ).add_to(m_choropleth)

folium.LayerControl().add_to(m_choropleth)

m_choropleth


Statistics by governorate:
   Governorate  Listings  Median_Price    Mean_Price
0        Ajlun        13       5076.00   9425.307692
1        Amman      1494      17625.00  24602.454839
2        Aqaba         8       9623.25  11006.812500
3        Balqa        27       9165.00  11953.666667
4        Irbid       208      15333.75  17752.266868
5       Jarash        16       9870.00  12500.531250
6        Karak        11      13959.00  11767.090909
7        Ma`an         1       8883.00   8883.000000
8       Madaba        18       9975.75  10101.083333
9       Mafraq        20       5322.75   6359.100000
10       Other        17       3031.50  10610.250000
11     Tafilah         6       5992.50   5593.000000
12       Zarqa       254      21925.50  22911.789449


While this type of visualization was not discussed during our lectures, I searched online and found that Folium offers an interesting alternative to Plotly for creating interactive maps. Unlike Plotly which focuses on scatter-based representations, Folium allows us to create choropleth maps that color entire administrative regions based on aggregated data, providing a different perspective on geographic patterns.

From the Folium map above, we can see the Jordanian car market visualized at the governorate level rather than by individual cities. The choropleth layer uses a YlOrRd color scale, where darker red shades indicate governorates with more car listings. Amman Governorate appears in the darkest red, confirming its dominant position with 1,494 listings. Zarqa Governorate shows as the next darkest, reflecting its 254 listings, followed by Irbid Governorate with 199 listings. The remaining governorates such as Balqa, Madaba, Jerash, Ajlun, Karak, Mafraq, Tafilah, Aqaba, and Ma'an appear in much lighter yellow-orange shades, indicating their relatively small share of the market, with each contributing less than 30 listings.

What makes this Folium visualization particularly valuable is that it shows the administrative boundaries of each governorate, giving context to the geographic distribution of car listings across the entire country. We can see that the market is heavily concentrated in the central and northern governorates, while the southern governorates like Aqaba, Ma'an, and Tafilah have very few listings despite covering large land areas. This likely reflects population distribution, economic activity, and the fact that most car dealerships and importers are based in Amman, Zarqa, and Irbid.

Additionally, the map includes circle markers for individual cities, sized proportionally to their listing volumes and colored in blue. This dual-layer approach allows us to see both the governorate-level aggregation and the specific city-level detail simultaneously. The interactive tooltips provide instant access to the data when hovering over each governorate or clicking on city markers. Overall, this Folium visualization complements the Plotly map by adding administrative boundary context and offering a different interactive experience that some viewers may find more intuitive for understanding regional patterns.

In [228]:
# Bar Chart: Top Cities by Listings and Price

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig_cities = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Top 10 Cities by Number of Listings', 'Top 10 Cities by Median Price'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}]]
)

top_listings = city_stats.head(10).sort_values('Listings', ascending=True)
fig_cities.add_trace(
    go.Bar(
        x=top_listings['Listings'],
        y=top_listings['City'],
        orientation='h',
        marker_color='#2196F3',
        name='Number of Listings',
        text=top_listings['Listings'],
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Listings: %{x}<extra></extra>'
    ),
    row=1, col=1
)

top_price = city_stats[city_stats['Listings'] >= 3].nlargest(10, 'Median_Price').sort_values('Median_Price', ascending=True)
fig_cities.add_trace(
    go.Bar(
        x=top_price['Median_Price'],
        y=top_price['City'],
        orientation='h',
        marker_color='#4CAF50',
        name='Median Price (USD)',
        text=top_price['Median_Price'].apply(lambda x: f'${x:,.0f}'),
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Median Price: $%{x:,.0f}<extra></extra>'
    ),
    row=1, col=2
)

fig_cities.update_layout(
    title='<b>Jordan Car Market: City Analysis</b>',
    height=500,
    showlegend=False,
    template='plotly_white'
)

fig_cities.update_xaxes(title_text='Number of Listings', row=1, col=1)
fig_cities.update_xaxes(title_text='Median Price (USD)', row=1, col=2)
fig_cities.update_yaxes(title_text='', row=1, col=1)
fig_cities.update_yaxes(title_text='', row=1, col=2)

fig_cities.show()

Here, the left chart shows that Amman dominates with nearly 1,500 listings, followed by Zarqa with about 250 and Irbid with about 200. After these top three, there is a sharp drop-off, with all other cities having fewer than 30 listings each. This confirms how heavily centralized the Jordanian car market is around Amman, Zarqa, and Irbid.

The right chart shows median prices by city. Interestingly, Zarqa has the highest median price at approximately USD 21,900, surpassing Amman at $17,600. This is surprising because Zarqa is often considered an industrial city, yet it commands higher prices than the capital. Irbid follows at USD 14,100, while the smaller cities like Salt, Madaba, and Jerash range between USD 9,000 and USD 10,000. The lowest prices are found in Ajloun, Al Mafraq, and Mafraq, ranging from USD 5,100 down to USD 4,700, which likely reflects older vehicle fleets in these areas.

In [229]:
#Price Distribution by City (Box Plot)

import plotly.express as px

# Selecting major cities for comparison based on the actual data
major_cities = ['Amman', 'Zarqa', 'Irbid', 'Salt', 'Madaba', 'Jerash', 'Ajloun', 'Karak']
df_cities_compare = df_jordan[df_jordan['City'].isin(major_cities)].copy()

print(f"Comparing {len(df_cities_compare)} listings from {len(major_cities)} cities")

fig_box = px.box(
    df_cities_compare,
    x='City',
    y='Price_USD',
    color='City',
    title='<b>Price Distribution by Major City</b><br><sub>Comparing price ranges across Jordanian cities</sub>',
    labels={'Price_USD': 'Price (USD)', 'City': 'City'},
    template='plotly_white',
    height=500,
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig_box.update_layout(showlegend=False)
fig_box.show()

print("\n" + "="*60)
print("PRICE SUMMARY BY MAJOR CITY")
print("="*60)
summary = df_cities_compare.groupby('City')['Price_USD'].agg(['count', 'median', 'mean', 'std']).round(2)
print(summary.to_string())

Comparing 2032 listings from 8 cities



PRICE SUMMARY BY MAJOR CITY
        count    median      mean       std
City                                       
Ajloun     13   5076.00   9425.31   7143.75
Amman    1494  17625.00  24602.45  23237.18
Irbid     199  14100.00  17025.70  15407.33
Jerash     16   9870.00  12500.53   9269.85
Karak      11  13959.00  11767.09   6391.63
Madaba     18   9975.75  10101.08   4943.87
Salt       27   9165.00  11953.67  11334.19
Zarqa     254  21925.50  22911.79  12559.92


From the box plot and summary statistics above, we can see how prices vary across major Jordanian cities. Amman has the largest number of listings at 1,494, with a median price of USD 17,625 and a mean of USD 24,602, indicating that the distribution is right-skewed, meaning there are some very expensive cars pulling the average upward. The standard deviation of USD 20,781 shows wide price variation in the capital.

Zarqa stands out with the highest median price at USD 21,925 and the highest mean at USD 22,912, with a relatively low standard deviation of USD 14,404. This suggests that Zarqa's market is more consistent and generally more expensive than Amman, possibly due to a higher concentration of newer or premium vehicles.

Irbid has a median of USD 14,100 and a mean of USD 17,026, with 199 listings, making it more affordable than both Amman and Zarqa. The smaller cities like Salt, Madaba, Jerash, Ajloun, and Karak have much fewer listings, ranging from 27 down to 11, and their median prices are significantly lower, between USD 5,076 and USD 9,975. Their standard deviations are also smaller, indicating less price variation, likely because these markets consist of older, cheaper vehicles with fewer luxury options available.

Overall, the box plot clearly shows that Amman and Zarqa have the highest and most spread-out price ranges, while smaller cities have tighter, lower-price distributions, reflecting different market characteristics and economic conditions across Jordan.


### Dashboard Direction
The next step is a **Dash/Plotly dashboard** with:
- A **filter panel** (Brand, Year, Fuel Type, City)
- **KPI cards** (median price, total listings, avg mileage)
- The **map** as the centrepiece
- A **comparison toggle** to switch between Jordan-only and Jordan vs USA views